# Exploring the UKPN dispatch dataRun each cell with **Shift+Enter**. Edit and re-run freely — nothing here writes to disk.The pipeline must have run first (`python -m ukpn.pipeline`).

In [ ]:
import syssys.path.insert(0, "../src")import duckdbimport pandas as pdpd.set_option("display.max_columns", 200)pd.set_option("display.width", 250)pd.set_option("display.float_format", lambda v: f"{v:,.3f}")con = duckdb.connect()for name in ["tidy", "validation_summary", "exceptions", "company_league",             "share_over_time", "price_by_group", "zone_concentration",             "direction_over_time", "product_mix"]:    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('../data/processed/{name}.parquet')")con.execute("SELECT count(*) AS dispatches FROM tidy").df()

## 1. What's in the table`SUMMARIZE` is DuckDB's version of profiling every column at once — min, max, distinct count, nulls, percentiles.Closest thing to opening a table in a GUI and eyeballing it.

In [ ]:
con.execute("SUMMARIZE tidy").df()

## 2. Which checks are firingThis is the blocker. If two or three checks dominate, they're probably measuring my assumptions rather than UKPN's data.

In [ ]:
con.execute("SELECT * FROM validation_summary ORDER BY rows_flagged DESC").df()

### How many rows fail *only* one check?A row failing a single check is usually a genuine edge case.A row failing five is usually one underlying cause tripping several checks at once.

In [ ]:
con.execute("""    SELECT n_flags, count(*) AS rows    FROM tidy_flags    GROUP BY 1 ORDER BY 1""").df() if "tidy_flags" in [t[0] for t in con.execute("SHOW TABLES").fetchall()] else pd.read_parquet("../data/processed/exceptions.parquet")["n_flags"].value_counts().sort_index().to_frame("rows")

## 3. The technology vocabulary questionDo the two naming conventions split on a date, or run in parallel?A clean split means UKPN changed taxonomy. An overlap means it's provider-specific reporting.

In [ ]:
con.execute("""    SELECT technology_vocabulary,           min(start_time) AS first_seen,           max(start_time) AS last_seen,           count(*)        AS dispatches,           count(DISTINCT technology) AS distinct_labels    FROM tidy    GROUP BY 1""").df()

In [ ]:
con.execute("""    SELECT date_trunc('quarter', start_time) AS quarter,           sum(CASE WHEN technology_vocabulary = 'lc31e' THEN 1 ELSE 0 END)       AS lc31e,           sum(CASE WHEN technology_vocabulary = 'descriptive' THEN 1 ELSE 0 END) AS descriptive    FROM tidy    GROUP BY 1 ORDER BY 1""").df()

### Does the vocabulary track the provider rather than the date?

In [ ]:
con.execute("""    SELECT company,           count(DISTINCT technology_vocabulary) AS vocabularies_used,           count(*) AS dispatches    FROM tidy GROUP BY 1    HAVING count(DISTINCT technology_vocabulary) > 1    ORDER BY dispatches DESC""").df()

## 4. The marketProvider names are published by UKPN, so this is the market as reported — not inferred.

In [ ]:
con.execute("SELECT * FROM company_league LIMIT 15").df()

### Share of volume over time, by technology group

In [ ]:
share = con.execute("""    SELECT date_trunc('quarter', start_time) AS quarter,           tech_group,           sum(utilisation_mwh) AS mwh    FROM tidy GROUP BY 1,2""").df()pivot = share.pivot(index="quarter", columns="tech_group", values="mwh").fillna(0)(100 * pivot.div(pivot.sum(axis=1), axis=0)).round(1)

In [ ]:
(100 * pivot.div(pivot.sum(axis=1), axis=0)).plot(figsize=(11,5), title="Share of requested MWh by technology group (%)")

## 5. Free-formYour scratchpad. `con.execute("...").df()` runs any SQL against any of the views.

In [ ]:
con.execute("""    SELECT product,           count(*) AS dispatches,           round(avg(hours_requested), 2)    AS avg_hours,           round(median(utilisation_price), 2) AS median_price,           round(sum(utilisation_mwh), 1)   AS mwh    FROM tidy    GROUP BY 1 ORDER BY dispatches DESC""").df()